In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model

print("✅ TensorFlow version:", tf.__version__)
print("✅ Keras import successful!")


✅ TensorFlow version: 2.15.0
✅ Keras import successful!


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from ultralytics import YOLO
from transformers import AutoConfig, SiglipForImageClassification, AutoImageProcessor
import cv2
import numpy as np
import os
import sys

# --- 1. PyTorch Model Definitions (Age and Gender) ---

# CNN for Binary Gender Classification
class GenderClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super(GenderClassifier, self).__init__()
        # Feature Extraction Layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.5)
        # Fully Connected Layers (Assuming 64x64 input -> 8x8x128 feature map)
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 8 * 8)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# CNN for Age Regression
class AgeRegressor(nn.Module):
    def __init__(self):
        super(AgeRegressor, self).__init__()
        # Feature Extraction Layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.5)
        # Fully Connected Layers
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 1) # Single output for age regression

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 8 * 8)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- 2. ModelSelector Class ---

class ModelSelector:
    def __init__(self, model_type):
        self.model_type = model_type
        # Set device to CPU for robust webcam performance
        self.device = torch.device("cpu")

        # Define 64x64 preprocessing once for Age/Gender models
        self.pyt_preprocess_64 = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((64, 64)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        if self.model_type == "sign":
            self._load_sign_model()
        elif self.model_type == "emotion":
            self._load_emotion_model()
        elif self.model_type == "gender":
            self._load_gender_model()
        elif self.model_type == "age":
            self._load_age_model()
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")

    # --- PyTorch Loading Helper ---
    def _load_pyt_model(self, model_class, model_path, num_classes=None):
        if num_classes:
            model = model_class(num_classes=num_classes).to(self.device)
        else:
            model = model_class().to(self.device)

        if not os.path.exists(model_path):
             print(f"ERROR: Model file not found at {model_path}. Please train it first.")
             return None
             
        # Load weights onto CPU
        model.load_state_dict(torch.load(model_path, map_location=self.device), strict=False)
        model.eval()
        return model

    # --- Load Specific Models ---
    def _load_gender_model(self):
        print("🔹 Loading Gender Classifier...")
        # CRITICAL: Use raw string (r"...") for absolute paths
        MODEL_PATH = r'C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\gender_classifier_best.pth'
        self.class_names = ['female', 'male'] 
        self.model = self._load_pyt_model(GenderClassifier, MODEL_PATH, num_classes=len(self.class_names))
        if self.model: print("✅ Gender model loaded successfully")

    def _load_age_model(self):
        print("🔹 Loading Age Regressor...")
        # CRITICAL: Use raw string (r"...") for absolute paths
        MODEL_PATH = r'C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\age_regressor_best.pth'
        self.model = self._load_pyt_model(AgeRegressor, MODEL_PATH)
        if self.model: print("✅ Age model loaded successfully")
    
    def _load_sign_model(self):
        print("🔹 Loading Sign Language model...")
        
        # CRITICAL: Define a single, absolute path where ALL Sign Model files live
        SIGN_MODEL_DIR = r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\export\sign_model"
        WEIGHTS_PATH = os.path.join(SIGN_MODEL_DIR, "model_weights.pt")
        
        if not os.path.exists(os.path.join(SIGN_MODEL_DIR, "config.json")):
             print(f"ERROR: Sign Model config not found at {SIGN_MODEL_DIR}. Did you run the export steps in SignLanguageTeacherModel.ipynb?")
             self.model = None
             return

        try:
            # FIX: Use local_files_only=True and the correct base directory
            config = AutoConfig.from_pretrained(SIGN_MODEL_DIR, local_files_only=True)
            self.model = SiglipForImageClassification(config)
            
            state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")
            self.model.load_state_dict(state_dict, strict=False)
            self.model.eval()

            # FIX: Use local_files_only=True for the processor as well
            self.processor = AutoImageProcessor.from_pretrained(SIGN_MODEL_DIR, use_fast=False, local_files_only=True)
            self.label_map = {str(i): chr(65 + i) for i in range(26)}
            print("✅ Sign model loaded successfully")
        except Exception as e:
            print(f"ERROR: Failed to load Sign model locally.")
            print(e)
            self.model = None

    def _load_emotion_model(self):
        print("🔹 Loading Emotion model (YOLO weighted)...")
        # CRITICAL: Absolute path pointing directly to the TRAINED WEIGHTS file (.pt)
        YOLO_PATH = r"C:\Users\anike\OneDrive\Desktop\ds project\HumanPatterns\best.pt"
        try:
            # FIX: YOLO automatically loads weights when pointed to a .pt file
            self.model = YOLO(YOLO_PATH)
            print("✅ Emotion model loaded successfully (YOLO weighted)")
        except Exception as e:
            print(f"ERROR: Failed to load YOLO weighted model. Check the path: {YOLO_PATH}")
            self.model = None

    # --- Prediction Methods ---
    def predict(self, image):
        if self.model is None:
            return "N/A (Model Failed)"
            
        if self.model_type == "sign":
            image_pil = transforms.ToPILImage()(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            inputs = self.processor(images=image_pil, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                pred_idx = outputs.logits.argmax(-1).item()
            return self.label_map[str(pred_idx)]

        elif self.model_type == "emotion":
            results = self.model.predict(image, imgsz=224, verbose=False)

            if results and results[0].probs is not None:
                pred_idx = results[0].probs.argmax().item()
                return results[0].names[pred_idx]
            else:
                return "Unknown"
            
        elif self.model_type == "gender":
            face_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            input_tensor = self.pyt_preprocess_64(face_rgb).unsqueeze(0).to(self.device)
            with torch.no_grad():
                outputs = self.model(input_tensor)
                _, predicted_idx = torch.max(outputs.data, 1)
            return self.class_names[predicted_idx.item()].capitalize()
        
        elif self.model_type == "age":
            face_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            input_tensor = self.pyt_preprocess_64(face_rgb).unsqueeze(0).to(self.device)
            with torch.no_grad():
                predicted_age = self.model(input_tensor).item()
            return f"{predicted_age:.1f}"

In [3]:
class RAGRouter:
    def __init__(self):
        # Initialize Haar Cascade for face detection
        HAARCASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        self.face_cascade = cv2.CascadeClassifier(HAARCASCADE_PATH)
        if self.face_cascade.empty():
            print("⚠️ WARNING: Face Detector (Haar Cascade) not loaded. Age/Gender detection will rely on full frame analysis.")
            self.face_cascade = None
        else:
            print("✅ Face Detector loaded.")

        # Initialize Models
        self.sign_selector = ModelSelector("sign")
        self.emotion_selector = ModelSelector("emotion")
        self.gender_selector = ModelSelector("gender")
        self.age_selector = ModelSelector("age")

    def run_webcam(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("⚠️ Cannot open webcam.")
            return

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            display_frame = frame.copy()
            
            # Convert to grayscale for faster face detection
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            faces = []
            if self.face_cascade:
                faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

            if len(faces) > 0:
                # Process the largest face for Age/Gender (simplification)
                x, y, w, h = max(faces, key=lambda f: f[2] * f[3])

                # Extract and pad the face crop for PyTorch models
                margin = int(0.2 * w)
                x1 = max(0, x - margin)
                y1 = max(0, y - margin)
                x2 = min(frame.shape[1], x + w + margin)
                y2 = min(frame.shape[0], y + h + margin)
                
                face_crop = frame[y1:y2, x1:x2]

                cv2.rectangle(display_frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
                
                # Predict Age/Gender on the cropped face
                gender_pred = self.gender_selector.predict(face_crop)
                age_pred = self.age_selector.predict(face_crop)
            
            else:
                gender_pred = "N/A (No Face)"
                age_pred = "N/A (No Face)"
            
            # --- Sign/Emotion Prediction (Uses a resized version of the whole frame) ---
            resized_224 = cv2.resize(frame, (224, 224))
            
            # For sign and emotion, pass the resized frame as input
            sign_pred = self.sign_selector.predict(resized_224)
            emotion_pred = self.emotion_selector.predict(resized_224)


            # --- Display Predictions ---
            cv2.putText(display_frame, f"Sign: {sign_pred}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(display_frame, f"Emotion: {emotion_pred}", (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            cv2.putText(display_frame, f"Gender: {gender_pred}", (20, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(display_frame, f"Age: {age_pred} yrs", (20, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 165, 0), 2)

            cv2.imshow("Multi-Model Router", display_frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()

In [4]:
if __name__ == "__main__":
    router = RAGRouter()
    router.run_webcam()

✅ Face Detector loaded.
🔹 Loading Sign Language model...
✅ Sign model loaded successfully
🔹 Loading Emotion model (YOLO weighted)...
✅ Emotion model loaded successfully (YOLO weighted)
🔹 Loading Gender Classifier...
✅ Gender model loaded successfully
🔹 Loading Age Regressor...
✅ Age model loaded successfully
